# Feature Extraction

This notebook extracts and visualizes various oceanic plate properties (including: seafloor age, sediment thickness, carbon content, and convergence rates). It uses plate tectonic reconstruction models to understand how these parameters have changed over millions of years.

We begin by importing the required libraries:

In [ ]:
# =============================================================================
# IMPORT LIBRARIES AND DEPENDENCIES
# =============================================================================

from ipywidgets import interact
import os

# Geospatial and plotting libraries
import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately.tools import plate_isotherm_depth
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

# Custom analysis modules
# NOTE: Ensure the 'lib/' folder is downloaded and in the same directory!
from lib.main import *
from lib.water_thickness import calculate_water_thickness
from lib.slab_dip import calculate_slab_dip

# Load configuration parameters (e.g. paths, model names)
from parameters import parameters

### Setup
As defined in  `'parameters.py'`:
- Sets the name of the plate reconstruction model (e.g., "Muller2019")
- Sets analysis parameters: time range, temporal resolution, and grid resolution
- Defines input (for loading geological datasets)/output (for saving results) directories and filenames

`NOTE`: You can change the analysis settings (like time range, resolution, and file locations) in `parameters.py`, which is in the same folder as this notebook.
It's structured as a dictionary; look for entries like 'plate_model_name', 'timespan', and 'grid_resolution' to adjust values as needed.

In [ ]:
# Plate reconstruction model to use 
plate_model_name = parameters["plate_model_name"]

# Set up temporal analysis parameters
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]                # Oldest time to analyze
time_max = parameters["timespan"]["max"]                # Youngest time to analyze
# Create array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

grid_resolution = parameters["grid_resolution"]

# Input/output directories paths
plate_model_dir = parameters["plate_model_dir"]
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

# Paths to different types of geological data grids
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
spreadrate_dir = os.path.join(inputs_dir, "SpreadingRate")
sedthick_dir = os.path.join(inputs_dir, "SedimentThickness")
carbonate_dir = os.path.join(inputs_dir, "CarbonateThickness")
co2_dir = os.path.join(inputs_dir, "CrustalCO2")

nprocs = 16

### Feature Extraction

This section performs the core subduction zone analysis. It either loads pre-computed subduction data or calculates everything from scratch using plate reconstructions and geological grids.

If pre-computed data exists, it loads previously saved subduction zone data from a `.csv file`. Otherwise, it:
- Loads a global plate reconstruction model to simulate plate positions through time.
- Computes where and how fast plates are converging i.e. identifies subduction zones.
- Co-registers trenches with geological grids (age, sediments, etc.).
- Calculates derived properties:

    - Lithospheric thickness from seafloor age.
    - Water and carbon content in the subducting slab.
    - Slab dip angle and effective subducted layer thickness (accounts for trench geometry and plate tilt).
- Computes subduction fluxes (sediment, carbon & water)

In [ ]:
# =============================================================================
# FEATURE EXTRACTION AND ANALYSIS
# =============================================================================

# Load the plate tectonic reconstruction model
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

# Check if processed subduction data already exists
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    # Step 1: Calculate convergence rates and subduction zone locations
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        plate_reconstruction=plate_model,
        verbose=True,                     # Print progress updates
    )
    
    # Step 2: Extract geological properties at subduction zone locations
    subduction_data = run_coregister_ocean_rasters(
        nprocs=nprocs,
        times=time_steps,
        input_data=subduction_data,
        plate_reconstruction=plate_model,
        agegrid_dir=agegrid_dir,
        spreadrate_dir=spreadrate_dir,
        sedthick_dir=sedthick_dir,
        carbonate_dir=carbonate_dir,
        co2_dir=co2_dir,
        verbose=True,
    )
    
    # Step 3: Calculate thermal lithosphere thickness
    subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
        subduction_data["seafloor_age (Ma)"],  # Input: age of oceanic crust
        maxiter=100,
    )
    
    # Step 4: Calculate additional geological properties
    subduction_data = calculate_water_thickness(data=subduction_data)
    subduction_data = calculate_carbon(subduction_data)
    subduction_data = calculate_slab_flux(subduction_data)
    subduction_data = calculate_slab_dip(subduction_data)

    # Extract thickness of subducted slab
    subduction_data = extract_subducted_thickness(
        subduction_data,
        plate_reconstruction=plate_model,
        grid_resolution=grid_resolution,
    )
    
    # Step 5: Calculate fluxes of different materials into subduction zones
    subduction_data["sediment_flux (m^2/yr)"] = (
        subduction_data["sediment_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2  # Convert cm/yr to m/yr
    ).clip(0.0, np.inf) # Ensure no negative fluxes

    subduction_data["carbon_flux (t/m/yr)"] = (
        subduction_data["total_carbon_density (t/m^2)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["water_flux (m^2/yr)"] = (
        subduction_data["total_water_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    # Save processed data
    subduction_data.to_csv(subduction_data_filename, index=False)

This section prepares to visualize geological properties at subduction zones across different time steps:

- It creates a list of which quantities to plot.
- It loads tectonic plate boundaries from the plate reconstruction model.
- It sets up a Mollweide map projection, centered at 60°E longitude.

These plots help interpret the spatial distribution of subduction-related features through geological time.

In [ ]:
# =============================================================================
# PREPARE DATA FOR VISUALIZATION
# =============================================================================

subduction_data_columns = subduction_data.columns.tolist()

# Create list of features available for plotting (exclude coordinate/ID columns)
features_plot = subduction_data_columns.copy()
features_plot.remove('lon')                    # Longitude coordinate
features_plot.remove('lat')                    # Latitude coordinate  
features_plot.remove('age (Ma)')
features_plot.remove('subducting_plate_ID')
features_plot.remove('trench_plate_ID')

# Set up plotting utilities for plate reconstruction visualization
gplot = get_plot_topologies(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    plate_reconstruction=plate_model,
    filter_topologies=True,
)

# Define map projection (Mollweide provides good global view)
projection = ccrs.Mollweide(central_longitude=60)

### Visualisation

This section creates an interactive map that allows users to:
1. Select different time periods (geological ages)

2. Choose different subduction zone properties to visualize

3. View the results overlaid on seafloor age and plate boundary data

The visualization shows:

- Background: Seafloor age (colored by age, showing oceanic crust formation)
- Points: Subduction zone locations (colored by selected property)
- Lines: Plate boundaries, mid-ocean ridges, and subduction zones
- Vectors: Plate motion directions

In [ ]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    # Load data for the specified geological time
    agegrid_filename = f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)

    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]


    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )

    # Create color bar axes for the selected feature and seafloor age
    cax_feat = fig.add_axes([0.1, 0.12, 0.35, 0.02])
    cax_bg = fig.add_axes([0.55, 0.12, 0.35, 0.02])
    
    # Plot seafloor age as background (blue-purple color scheme)
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, 
                        vmin=0, vmax=230,
                        alpha=0.7, zorder=1)
    
    # Plot continental areas in dark gray
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, 
                                   normalise=True, alpha=0.1, zorder=3)

    # Plot subduction zone data points
    feat = ax.scatter(subduction_data_t['lon'], subduction_data_t['lat'], 
                     50, marker='.',
                     c=subduction_data_t[feature],
                     cmap=ccm.hawaii_r,
                     transform=ccrs.PlateCarree(),
                     zorder=4)
    
#     gplot.plot_all_topological_sections(ax, color='dimgray', linewidth=1.5, zorder=5)

    # Plot plate boundaries and tectonic features
    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)

#     gplot.plot_ridges_and_transforms(ax, color='dimgray', linewidth=1.5, zorder=5)

    # Highlight subduction zones specifically
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)           # Trench lines
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k',          # Triangular teeth
                               alpha=0.3, zorder=7)
    
    # Add coordinate grid lines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, 
                     x_inline=False, linewidth=1, color="gray", 
                     alpha=0.3, linestyle="--", zorder=8)
    
    # Manually add longitude labels
    ax.text(0.49, -0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46, -0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40, -0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    # Configure grid labels
    gl.top_labels = False
    gl.bottom_labels = False
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    # Create color bars with labels
    cbar_feat = fig.colorbar(feat, cax=cax_feat, orientation="horizontal", extend='both')
    cbar_feat.set_label(feature, fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend='max')
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])  # Major age divisions
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Create custom legend for geological features
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Line2D([0], [0], color='dimgray', lw=2, label='Plate Boundaries')
    ]
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', 
             bbox_to_anchor=(0, -0.15))
    
    # Add title showing the geological time
    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)

    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …